# Тема 8 — Production inference API: шлюз поверх замороженного профиля

## Пропуск: заказ → квитанция → окно выдачи

**Что случилось в Topic 07.** Вы заморозили production-профиль движка (`max_num_seqs=32`)
с прогнозом и планом отката. Но профиль — внутренняя кухня: клиент не может
«импортировать» его. Между чужим HTTP-запросом и вашим движком стоит шлюз.

**Проблема.** HTTP 200 — это не «успех», а «ответ отправлен». Закрытое соединение —
не отмена. Очередь без границ — не ёмкость, а задержка, спрятанная от пейджера.
Каждый механизм Topic 06 теперь возвращается как решение API: admission control — это
очередь из лабы насыщения; отключение клиента — история preemption; честный 503 —
таксономия терминальных исходов в статус-кодах.

**Что меняется сейчас.** Один тонкий FastAPI-gateway поверх движка. Мотив темы —
«пропуск»: заказ → квитанция → окно выдачи. Билет, который вы держите, либо готовится,
либо честно отказан — третьего не дано.

**Ваша задача на лабу.** (1) Заморозить таблицу предсказаний контракта ДО кода шлюза.
(2) Поднять v0-шлюз и получить первый round-trip по настоящему HTTP-стеку. (3) Встроить
валидацию капов, ограниченную очередь с честным 429 и карту ошибок. (4) Уронить шлюз
двумя drill'ами и прочитать журнал. (5) Обучить структурированный вывод с проверкой
схемы. (6) Научить шлюз отмене и прогнать через него волновой драйвер. Stretch: /metrics
и мемо контракта.

**Правила игры.** (1) Каждая клетка отвечает на один названный вопрос; вывод клетки —
доказательство ответа. (2) Числа мока — объявленная синтетика: ручки прописаны в клетке
движка и подписаны в тексте, это не измерения vLLM. (3) Живой vLLM за шлюзом —
демонстрация владельца, не студенческий шаг. (4) Предсказания замораживаются до
измерений — как в Topics 06–07.

Конверт: 2 контактных часа + 3–5 самостоятельных. Всё ядро лабы работает на бесплатном CPU.

In [ ]:
import asyncio, hashlib, itertools, json, os, socket, threading, time
import urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor
from typing import Sequence, TypedDict
from tqdm.auto import tqdm  # правило курса: видимый прогресс в каждой клетке дольше ~5 с

PROFILE = 'mock'  # весь контур шлюза исполняется на CPU: MockEngine, объявленная синтетика
ATTEMPT = 'attempt-001'
RESULTS = os.path.join('results', PROFILE, ATTEMPT)
os.makedirs(RESULTS, exist_ok=True)
PORT = 8018  # наш собственный процесс; teardown в конце ноутбука остановит его штатно
BASE = f'http://127.0.0.1:{PORT}'
print(f'profile={PROFILE}  results dir={RESULTS}  port={PORT}')

In [ ]:
# fastapi и uvicorn НЕ предустановлены в Colab — ставим до первого HTTP-вызова.
# Оценка времени: ~5–40 с (пакеты маленькие; до 3 попыток при сетевых сбоях).
# Итог печатается ОДНОЙ строкой — паттерн t07-gpu-bootstrap (счётчик попыток в финале).
import importlib.util, subprocess  # noqa: E402
_need = [p for p in ('fastapi', 'uvicorn') if importlib.util.find_spec(p) is None]
_attempts = 0
_out = None
for _attempt in range(3 if _need else 0):
    _attempts = _attempt + 1
    _out = subprocess.run(['pip', 'install', '-q', *_need, '--timeout', '120'],
                          capture_output=True, text=True)
    if _out.returncode == 0:
        break
    time.sleep(5)
_status = 'уже установлены' if not _need else (
    f'установлены с попытки {_attempts}' if _out is not None and _out.returncode == 0
    else 'ОШИБКА установки: ' + ((_out.stderr if _out is not None else '')[-200:]))
print(f'bootstrap: fastapi/uvicorn {_status}')

## Блок 0 — Контракт: предсказать ДО кода

Механизмы Topic 06 возвращаются как решения API. Пока не написано ни строчки кода
шлюза, заполните таблицу «ситуация → механизм → ожидаемый код» и заморозьте её в
`api-contract.json` — первый артефакт лабы, как прогноз колена в Topic 06.

| Ситуация клиента | Механизм из Topic 06 | Что честно ответить |
|---|---|---|
| запрос без `prompt`, кап превышен | планирование по факту, капы | отказ на входе, 400 |
| слотов нет, очередь полна | ограниченная очередь насыщения | 429 + `Retry-After` |
| клиент закрыл соединение | preemption брошенной работы | отмена: ответ не доставляется |
| движок лежит / бюджет вышел | терминальный исход | 503 / 504 с request id |

**Что вы должны увидеть в таблице ниже:** шесть строк, у каждой — названный механизм.
Две ячейки сознательно оставлены вам (уровень A): предсказать код для перегрузки и
для отмены.

**Вопрос на понимание:** почему «никогда не 200-with-error-body» — это не эстетика,
а контракт для клиента, у которого своя retry-логика?

In [ ]:
def predict_contract() -> list[dict]:
    """Freeze the gateway contract prediction table BEFORE any gateway code runs.

    Returns
    -------
    list of dict
        По строке на ситуацию клиента: situation, mechanism (механизм из Topic 06),
        predicted_code — ожидаемое поведение шлюза.
    """
    # Уровень A: почти готово — впишите два предсказания (TODO) своими словами.
    # Заморозка в api-contract.json идёт ДО любого кода шлюза: клетки движка и v0 ниже.
    return [
        dict(situation='запрос без обязательного поля prompt',
             mechanism='валидация на входе: планирование по факту',
             predicted_code=400),
        dict(situation='max_tokens = 999 при потолке 64',
             mechanism='жёсткий кап на входе — admission control до очереди',
             predicted_code=400),
        dict(situation='свободных слотов нет, очередь полна',
             mechanism='ограниченная очередь из лабы насыщения',
             predicted_code='TODO'),
        dict(situation='клиент закрыл соединение посреди генерации',
             mechanism='preemption: брошенная работа прекращается',
             predicted_code='TODO'),
        dict(situation='движок поднял флаг down',
             mechanism='терминальный исход: честный отказ без внутренностей',
             predicted_code=503),
        dict(situation='запрос ждёт в очереди дольше бюджета',
             mechanism='бюджет запроса: timeout из Topic 06',
             predicted_code=504),
    ]

_rows = predict_contract()
with open(os.path.join(RESULTS, 'api-contract.json'), 'w', encoding='utf-8') as fh:
    json.dump(_rows, fh, ensure_ascii=False, indent=1)
import pandas as pd  # таблица: живой замороженный контракт ДО любого кода шлюза
display(pd.DataFrame(_rows).style.set_caption(
    f'прогноз контракта заморожен ДО кода ({len(_rows)} строк → api-contract.json); '
    'сверим с журналами drills'))

## Справочник: термины блока

| Термин | Что это |
|---|---|
| gateway | тонкий HTTP-слой между клиентами и движком |
| endpoint | маршрут: `/health` (живость) и `/v1/completions` (основной) |
| admission control | решение «впустить / в очередь / отказать» на входе |
| backpressure | честное давление на клиента: 429 + `Retry-After` |
| request id | идентичность запроса: ответ, журнал, ошибка, метрики |
| timeout (deadline) | бюджет запроса; выход за него — 504 с id |
| cancellation | остановка брошенной генерации + учёт потраченных токенов |
| streaming (SSE) | кадры по мере генерации; отмена — тот же механизм, кадром позже |
| guided JSON | структурированный вывод: схема на входе, валидация на выходе |
| /metrics | счётчики: accepted, rejected-429, cancelled, in-flight |
| orphan | генерация, чей клиент ушёл, а движок досчитывает в никуда |

**Инфраструктура, которая уже написана за вас:** MockEngine (объявленные ручки
задержки), проводка шлюза `make_gateway`, запуск/остановка uvicorn в фоновом потоке,
stdlib-клиент `http_request` (никаких лишних зависимостей), журнал запросов.
Свёрнутые клетки — откройте и прочитайте: ничего магического.

In [ ]:
class EngineDown(RuntimeError):
    """Raised by the mock engine when the owner has flagged it down (drill 503)."""


class JournalRecord(TypedDict):
    """One gateway-observed request from arrival to terminal outcome."""
    request_id: str
    prompt_chars: int
    max_tokens: int
    wave: int
    arrived_ts: float
    finished_ts: float | None
    outcome: str
    http_status: int | None
    generated_tokens: int
    wasted_tokens: int


class MockEngine:
    """Async in-process mock of the pinned engine; declared synthetic latency.

    Полный прогон запроса занимает ttft_s + per_token_s * max_tokens секунд —
    это ОБЪЯВЛЕННАЯ модель мока, а не измерение vLLM: числа видны в ручках ниже
    и подписаны в тексте клеток.
    """

    def __init__(self, ttft_s: float = 0.15, per_token_s: float = 0.02,
                 max_concurrency: int = 32) -> None:
        """Bind the declared latency knobs for this mock engine instance.

        Parameters
        ----------
        ttft_s : float, default 0.15
            Declared seconds before the first token (ручка мока, не измерение).
        per_token_s : float, default 0.02
            Declared seconds per generated token (ручка мока).
        max_concurrency : int, default 32
            Ceiling on simultaneous generations — зеркало замороженного профиля
            seqs-32 из Topic 07; связывающий предел у шлюза ставит его очередь.

        Returns
        -------
        None
            Stateful helper; see generate().
        """
        self.ttft_s = ttft_s
        self.per_token_s = per_token_s
        self.down = False  # флаг drill'а «движок лежит» (t08-failure-drills)
        self._slots = asyncio.Semaphore(max_concurrency)
        self._running = 0
        self._started: dict[str, float] = {}
        self._caps: dict[str, int] = {}

    def note_start(self, request_id: str) -> None:
        """Register a generation start for the waste accounting.

        Parameters
        ----------
        request_id : str
            Gateway-assigned id of the request being started.

        Returns
        -------
        None
            Bookkeeping for tokens_so_far().
        """
        self._started[request_id] = time.monotonic()

    def active_count(self) -> int:
        """Count generations currently running inside the mock.

        Returns
        -------
        int
            Started-but-unfinished generations (виден в /health).
        """
        return self._running

    def tokens_so_far(self, request_id: str) -> int:
        """Estimate generated tokens of a running generation (declared mock accounting).

        Parameters
        ----------
        request_id : str
            Id registered by note_start().

        Returns
        -------
        int
            Tokens generated so far: (elapsed − ttft) / per_token, не больше капа.
        """
        started = self._started.get(request_id)
        if started is None:
            return 0
        elapsed = time.monotonic() - started - self.ttft_s
        cap = self._caps.get(request_id, 0)
        return max(0, min(cap, round(elapsed / self.per_token_s)))

    async def generate(self, request_id: str, prompt: str, max_tokens: int,
                       record: JournalRecord, structured: dict | None = None,
                       break_schema: bool = False) -> str:
        """Run one mock generation to completion and finalize its journal row.

        Parameters
        ----------
        request_id : str
            Gateway-assigned id, journaled verbatim.
        prompt : str
            Prompt text; the mock использует только длину.
        max_tokens : int
            Requested output-token cap; the mock emits exactly this many.
        record : JournalRecord
            Journal row, updated in place (outcome, tokens, timestamps).
        structured : dict or None, default None
            When set, the reply text is a mock JSON object for this schema
            (объявленная синтетика; настоящий grammar-constrained decode —
            на стороне живого движка).
        break_schema : bool, default False
            Fixture flag: the structured reply deliberately breaks the schema
            (путь violations в t08-guided-todo).

        Returns
        -------
        str
            Reply text; для structured-запроса — JSON-строка mock-объекта.

        Raises
        ------
        EngineDown
            Когда владелец поднял флаг down (drill честного 503).
        """
        if self.down:
            raise EngineDown('mock engine flagged down by the drill')
        self._caps[request_id] = max_tokens
        total_s = self.ttft_s + self.per_token_s * max_tokens
        self._running += 1
        try:
            async with self._slots:
                await asyncio.sleep(total_s)
        finally:
            self._running -= 1
        record['generated_tokens'] = max_tokens
        record['wasted_tokens'] = 0
        record['outcome'] = 'completed'
        record['finished_ts'] = time.monotonic()
        if structured is not None:
            if break_schema:
                return json.dumps({'label': 'оплата', 'priority': 'срочно',
                                   'score': '0.9'}, ensure_ascii=False)
            return json.dumps({'label': 'оплата', 'priority': 'высокий',
                               'score': 0.9}, ensure_ascii=False)
        return 'SYNTHETIC-MOCK-COMPLETION'


In [ ]:
from fastapi import FastAPI, Request  # noqa: E402
from fastapi.responses import JSONResponse  # noqa: E402
from pydantic import BaseModel  # noqa: E402
import uvicorn  # noqa: E402

REQUEST_BUDGET_S = 12.0  # объявленный бюджет запроса в очереди шлюза (ручка, не измерение)
JOURNAL: list[JournalRecord] = []
_REQUEST_SEQ = itertools.count(1)
_DEFAULT_MAP = dict(
    invalid=(400, {'error': 'invalid request'}, {}),
    overloaded=(429, {'error': 'queue full'}, {'Retry-After': '1'}),
    engine_down=(503, {'error': 'engine down'}, {}),
    deadline=(504, {'error': 'gateway timeout'}, {}),
    engine_bug=(500, {'error': 'internal error'}, {}))


class CompletionBody(BaseModel):
    """One completion request: prompt, token cap, optional guided schema."""
    prompt: str
    max_tokens: int = 16
    response_schema: dict | None = None


def _answer(outcome: str, request_id: str, map_error_fn: object) -> tuple[int, dict, dict]:
    """Resolve one terminal outcome to (status, payload, headers) for the client.

    Parameters
    ----------
    outcome : str
        Terminal outcome: 'invalid', 'overloaded', 'engine_down', 'deadline', 'engine_bug'.
    request_id : str
        Gateway-assigned id; travels inside every error payload.
    map_error_fn : object
        Student function map_engine_error or None for the default map.

    Returns
    -------
    tuple of (int, dict, dict)
        HTTP status, JSON payload and extra headers for JSONResponse.
    """
    if map_error_fn is not None:
        mapped = map_error_fn(outcome, request_id)
        return mapped['http_status'], mapped['payload'], mapped.get('headers', {})
    status, payload, headers = _DEFAULT_MAP[outcome]
    return status, dict(payload, request_id=request_id), dict(headers)


def make_gateway(engine: MockEngine, queue: 'AdmissionQueue | None' = None,
                 validate_fn: object = None, map_error_fn: object = None,
                 watch_fn: object = None, guided_fn: object = None) -> FastAPI:
    """Wire one gateway app around the engine from the pieces built so far.

    Проводка предоставлена курсом: каждый None просто выключает свой этап пропуска.
    Студенческие функции (validate_request, AdmissionQueue.admit, map_engine_error,
    watch_disconnect, guided_completion) подключаются сюда по мере сборки.

    Parameters
    ----------
    engine : MockEngine
        The backend behind the gateway (мок с объявленными ручками).
    queue : AdmissionQueue or None, default None
        Bounded admission queue; None — впускать всё (только v0).
    validate_fn : object
        validate_request(body) -> str | None; None — без валидации.
    map_error_fn : object
        map_engine_error(outcome, id) -> dict; None — карта по умолчанию.
    watch_fn : object
        watch_disconnect(...) coroutine; None — брошенные генерации не останавливаются.
    guided_fn : object
        guided_completion(data, schema) -> dict; None — схемная проверка выключена.

    Returns
    -------
    FastAPI
        The ASGI app для start_gateway().
    """
    app = FastAPI(title='t08-gateway')

    @app.get('/health')
    async def health() -> dict:
        """Report liveness, in-flight count and admission counters.

        Returns
        -------
        dict
            Keys: status, in_flight, counters — зонд живости и healthcheck.
        """
        return dict(status='ok', in_flight=engine.active_count(),
                    counters=dict(queue.counters) if queue is not None else {})

    @app.post('/v1/completions')
    async def completions(body: CompletionBody, request: Request) -> JSONResponse:
        """Serve one completion through validate → admit → engine → map.

        Parameters
        ----------
        body : CompletionBody
            Decoded request: prompt, max_tokens, optional response_schema.
        request : Request
            Raw ASGI request; заголовки несут тестовые крючки: x-wave,
            x-disconnect-after-s, x-mock-break-schema.

        Returns
        -------
        JSONResponse
            200 с квитанцией (заголовок X-Request-Id) или честный код из карты
            исходов; никогда не 200-with-error-body.
        """
        rid = f'r{next(_REQUEST_SEQ)}'
        record = JournalRecord(request_id=rid, prompt_chars=len(body.prompt),
                               max_tokens=body.max_tokens, wave=-1,
                               arrived_ts=time.monotonic(), finished_ts=None,
                               outcome='accepted', http_status=None,
                               generated_tokens=0, wasted_tokens=0)
        wave_header = request.headers.get('x-wave')
        if wave_header is not None:
            record['wave'] = int(wave_header)
        JOURNAL.append(record)
        if validate_fn is not None:
            reason = validate_fn(body.model_dump())
            if reason is not None:
                record.update(outcome='invalid', http_status=400, finished_ts=time.monotonic())
                return JSONResponse(status_code=400, content=dict(error=reason, request_id=rid))
        verdict = 'admit'
        if queue is not None:
            verdict = queue.admit(rid)
            if verdict == 'reject':
                record.update(outcome='overloaded', finished_ts=time.monotonic())
                status, payload, headers = _answer('overloaded', rid, map_error_fn)
                record['http_status'] = status
                return JSONResponse(status_code=status, content=payload, headers=headers)
            if verdict == 'queue':
                try:
                    await asyncio.wait_for(queue.wait_turn(rid), timeout=REQUEST_BUDGET_S)
                except asyncio.TimeoutError:
                    queue.abandon(rid)
                    record.update(outcome='deadline', finished_ts=time.monotonic())
                    status, payload, headers = _answer('deadline', rid, map_error_fn)
                    record['http_status'] = status
                    return JSONResponse(status_code=status, content=payload, headers=headers)
        watcher = None
        disconnect = asyncio.Event()
        try:
            engine.note_start(rid)
            engine_task = asyncio.create_task(engine.generate(
                rid, body.prompt, body.max_tokens, record,
                structured=body.response_schema,
                break_schema=request.headers.get('x-mock-break-schema') == '1'))
            if watch_fn is not None and 'x-disconnect-after-s' in request.headers:
                delay = float(request.headers['x-disconnect-after-s'])

                async def _fire_later() -> None:
                    """Set the disconnect event after the declared header delay.

                    Returns
                    -------
                    None
                        Событие выставляется по истечении объявленной задержки.
                    """
                    await asyncio.sleep(delay)
                    disconnect.set()

                asyncio.create_task(_fire_later())
                watcher = asyncio.create_task(
                    watch_fn(rid, engine_task, disconnect, record, engine))
            _done, pending = await asyncio.wait({engine_task}, timeout=REQUEST_BUDGET_S + 5.0)
            for task in pending:
                task.cancel()
            if engine_task in pending and not (watcher is not None and disconnect.is_set()):
                record.update(outcome='deadline', finished_ts=time.monotonic(),
                              wasted_tokens=engine.tokens_so_far(rid))
                status, payload, headers = _answer('deadline', rid, map_error_fn)
                record['http_status'] = status
                return JSONResponse(status_code=status, content=payload, headers=headers)
            if engine_task in pending or engine_task.cancelled():
                # клиент ушёл: наблюдатель остановил генерацию и финализировал журнал
                record['http_status'] = 504
                return JSONResponse(status_code=504, content=dict(
                    error='client gone; generation cancelled', request_id=rid))
            failure = engine_task.exception()
            if failure is not None:
                outcome = 'engine_down' if isinstance(failure, EngineDown) else 'engine_bug'
                record.update(outcome=outcome, finished_ts=time.monotonic())
                status, payload, headers = _answer(outcome, rid, map_error_fn)
                record['http_status'] = status
                return JSONResponse(status_code=status, content=payload, headers=headers)
            record['http_status'] = 200
            payload = dict(request_id=rid, text=engine_task.result(),
                           usage=dict(prompt_tokens=len(body.prompt.split()),
                                      completion_tokens=body.max_tokens))
            if body.response_schema is not None and guided_fn is not None:
                data = json.loads(engine_task.result())
                check = guided_fn(data, body.response_schema)
                if not check['ok']:
                    record.update(outcome='invalid')
                    return JSONResponse(status_code=422, content=dict(
                        error='guided reply violates the schema',
                        violations=check['violations'], request_id=rid))
                payload['data'] = data
            response = JSONResponse(status_code=200, content=payload)
            response.headers['X-Request-Id'] = rid
            return response
        finally:
            if watcher is not None and not watcher.done():
                watcher.cancel()
            if queue is not None and verdict in ('admit', 'queue'):
                queue.release(rid)
    return app




In [ ]:
def start_gateway(app: FastAPI, port: int) -> tuple[uvicorn.Server, threading.Thread]:
    """Run one gateway app under uvicorn in a background thread and wait for /health.

    Parameters
    ----------
    app : FastAPI
        The ASGI application to serve (наш собственный процесс).
    port : int
        Local port to bind on 127.0.0.1.

    Returns
    -------
    tuple of (uvicorn.Server, threading.Thread)
        The server (для штатного should_exit в teardown) and its thread.
    """
    config = uvicorn.Config(app, host='127.0.0.1', port=port, log_level='warning')
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    deadline = time.monotonic() + 30
    with tqdm(total=30, desc='ждём /health шлюза', unit=' с') as bar:
        while not server.started:
            assert time.monotonic() < deadline, 'шлюз не поднялся за 30 с'
            time.sleep(0.2)
            bar.update(0.2)
    return server, thread


def stop_gateway(server: uvicorn.Server, thread: threading.Thread) -> None:
    """Shut one owned gateway down gracefully and join its thread.

    Parameters
    ----------
    server : uvicorn.Server
        Server object returned by start_gateway().
    thread : threading.Thread
        Its background thread.

    Returns
    -------
    None
        should_exit = True — штатная остановка собственного процесса.
    """
    server.should_exit = True
    thread.join(timeout=15)


def http_request(url: str, body: dict | None = None, timeout_s: float = 30.0,
                 headers: dict | None = None) -> tuple[int, dict]:
    """Send one stdlib-urllib HTTP request and return (status, parsed JSON).

    Parameters
    ----------
    url : str
        Full URL, e.g. BASE + '/v1/completions'.
    body : dict or None, default None
        When set, the request is POST with this JSON payload.
    timeout_s : float, default 30.0
        Wall budget for the whole call.
    headers : dict or None, default None
        Extra headers (тестовые крючки: x-wave, x-disconnect-after-s).

    Returns
    -------
    tuple of (int, dict)
        HTTP status (0 — соединение умерло вместе с уходящим клиентом) and
        the parsed body ({}, когда тела нет).
    """
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(url, data=data,
                                 headers={'Content-Type': 'application/json',
                                          **(headers or {})})
    try:
        with urllib.request.urlopen(req, timeout=timeout_s) as resp:
            raw = resp.read()
            return resp.status, (json.loads(raw) if raw else {})
    except urllib.error.HTTPError as err:
        raw = err.read()
        try:
            return err.code, (json.loads(raw) if raw else {})
        except json.JSONDecodeError:
            return err.code, {}
    except (urllib.error.URLError, OSError):
        return 0, {}


def leave_after_send(path: str, body_bytes: bytes, headers: dict | None = None) -> None:
    """Deliver one request and slam the connection shut like a gone client.

    Parameters
    ----------
    path : str
        Request path, e.g. '/v1/completions'.
    body_bytes : bytes
        Encoded JSON body to deliver before closing the socket.
    headers : dict or None, default None
        Extra headers (тестовый крючок x-disconnect-after-s для drill'а отмены).

    Returns
    -------
    None
        Ответ никогда не читается: клиент «ушёл» сразу после отправки.
    """
    sock = socket.create_connection(('127.0.0.1', PORT), timeout=5)
    extra = ''.join(f'{k}: {v}\r\n' for k, v in (headers or {}).items())
    head = ('POST ' + path + ' HTTP/1.1\r\nHost: localhost\r\n' + extra
            + 'Content-Type: application/json\r\nContent-Length: '
            + str(len(body_bytes)) + '\r\nConnection: close\r\n\r\n').encode()
    sock.sendall(head + body_bytes)
    sock.close()


In [ ]:
# Первый работающий артефакт: v0-шлюз = голый пропуск запроса в движок.
# Никакой валидации, очереди и карты ошибок — они появятся в блоках ниже.
engine = MockEngine()
server, thread = start_gateway(make_gateway(engine), PORT)
_hstatus, _health = http_request(BASE + '/health')
_cstatus, _reply = http_request(BASE + '/v1/completions',
                                body={'prompt': 'Скажи одно слово: работает',
                                      'max_tokens': 8})
print(f"v0-пропуск: /health → {_hstatus} {dict(_health, counters='…')}; "
      f"completion → {_cstatus}, usage={_reply['usage']} (полный прогон мока: "
      f'{engine.ttft_s + engine.per_token_s * 8:.2f} с — объявленная арифметика)')

In [ ]:
# Данные, которыми занимается шлюз: живые запросы и живые ответы по HTTP
# (принцип курса: данные раньше формул). Промпты — формат замороженного корпуса.
SAMPLES = [
    dict(case_id='t08-a1',
         prompt='Тикет A: пользователь не видит кнопку оплаты. Классифицируй одним словом.',
         max_tokens=8),
    dict(case_id='t08-a2',
         prompt='Тикет A: приложение падает при загрузке аватара. Классифицируй одним словом.',
         max_tokens=8),
    dict(case_id='t08-b1',
         prompt='Документ B: регламент обслуживания терминалов, 3 абзаца. Резюме одним предложением.',
         max_tokens=24),
]
_rows = []
for _s in SAMPLES:
    _t0 = time.monotonic()
    _status, _reply = http_request(BASE + '/v1/completions',
                                   body={'prompt': _s['prompt'],
                                         'max_tokens': _s['max_tokens']})
    _rows.append(dict(case_id=_s['case_id'], http_status=_status,
                      wall_s=round(time.monotonic() - _t0, 2),
                      out_tokens=_reply['usage']['completion_tokens'],
                      reply=_reply['text'][:28]))
import pandas as pd
display(pd.DataFrame(_rows).style.set_caption(
    'живой round-trip через v0: ответы мока — объявленная SYNTHETIC-синтетика, не vLLM'))

## Блок 1 — Контракт на входе: капы и идентичность

Схема запроса — это обещание с потолками, а не пожелание: `prompt` не длиннее 512
символов, `max_tokens` в диапазоне 1..64. Валидация на входе — это admission control
ДО очереди: плохой запрос не должен занимать даже слот ожидания.

**Идентичность.** Каждый запрос получает id на входе, и id путешествует по всем
четырём местам: заголовок ответа `X-Request-Id`, журнал движка, тело ошибки, /metrics.
Без него сбой в 3 ночи необъясним.

**Вопрос клетке:** четыре зонда — валидный запрос, промпт длиннее капа, `max_tokens`
= 999, пустой промпт. Ровно один должен пройти. Какой и почему?

In [ ]:
PROMPT_CHAR_CAP = 512
MAX_TOKENS_CAP = 64


def validate_request(body: dict) -> str | None:
    """Return None for an admissible request, else a short rejection reason.

    Parameters
    ----------
    body : dict
        Decoded request body: prompt, max_tokens, optional response_schema.

    Returns
    -------
    str or None
        None — валидно, пустить дальше; строка — причина отказа (станет 400).
    """
    # Уровень A: почти готово. Проверка промпта уже написана — впишите 1–2 строки:
    # max_tokens должен быть целым в диапазоне 1..MAX_TOKENS_CAP.
    prompt = body.get('prompt')
    if not isinstance(prompt, str) or not prompt.strip():
        return 'prompt обязателен и должен быть непустой строкой'
    if len(prompt) > PROMPT_CHAR_CAP:
        return f'prompt длиннее потолка {PROMPT_CHAR_CAP} символов'
    # TODO: проверьте max_tokens и верните причину отказа
    return None


_probes = [
    dict(prompt='привет', max_tokens=8, expect=None),
    dict(prompt='x' * 600, max_tokens=8, expect='причина'),
    dict(prompt='привет', max_tokens=999, expect='причина'),
    dict(prompt='', max_tokens=8, expect='причина'),
]
_rows = [dict(prompt_chars=len(p['prompt']), max_tokens=p['max_tokens'],
              verdict=validate_request(dict(prompt=p['prompt'], max_tokens=p['max_tokens'])),
              expected=p['expect']) for p in _probes]
import pandas as pd
display(pd.DataFrame(_rows).style.set_caption(
    'зонд капов: валидация на входе — admission control до очереди; ровно один None'))

## Блок 2 — Ограниченная очередь: экономика глубины

Неограниченная очередь — задержка, которую вы пообещали никому. Глубина подчиняется
закону Литтла: **L = λ × W** — среднее число запросов в системе равно интенсивности
прихода, умноженной на среднее время пребывания.

**Объявленная арифметика мока** (ручки — в клетке движка): W = 0.15 + 0.02 × 16 =
0.47 с на запрос при 16 токенах. Потолок обслуживания c = 4 слота, буфер q = 4 места
ожидания — вместе 8. Колено: λ* = 8 / 0.47 ≈ 17.0 req/s. Дальше — 429 с
`Retry-After`: честный ответ вместо вечного ожидания.

**Конверт drill'а (заморожен до запуска):** бурст 24 клиента, запросы по 64 токена
(0.15 + 0.02 × 64 = 1.43 с): ждём 8 завершений (4 слота + 4 из очереди) и ровно
16 отказов 429. Расхождение с конвертом — самые интересные данные: назовите механизм.

**Вопрос:** почему 429 + `Retry-After` честнее длинной очереди, если клиент всё
равно повторит запрос?

In [ ]:
class AdmissionQueue:
    """Bounded admission: capacity in service, queue_slots waiting, the rest refused."""

    def __init__(self, capacity: int = 4, queue_slots: int = 4) -> None:
        """Bind the pool sizes for this admission queue.

        Parameters
        ----------
        capacity : int, default 4
            Service slots (запросы, которые шлюз держит в движке прямо сейчас).
        queue_slots : int, default 4
            Bounded waiting slots (места в очереди; сверх — честный отказ).

        Returns
        -------
        None
            Stateful helper; see admit().
        """
        self.capacity = capacity
        self.queue_slots = queue_slots
        self._lock = threading.Lock()
        self._active: set[str] = set()
        self._waiting: dict[str, float] = {}
        self._events: dict[str, asyncio.Event] = {}
        self.counters = dict(admitted=0, queued=0, rejected=0)

    def admit(self, request_id: str) -> str:
        """Decide admission for one request under the bounded-queue policy.

        Parameters
        ----------
        request_id : str
            Gateway-assigned id of the request.

        Returns
        -------
        str
            'admit' — слот обслуживания взят; 'queue' — место в очереди взято;
            'reject' — всё полно, вызывающий ответит 429.
        """
        # Уровень B: скелет готов — заполните шаги под lock.
        # Шаг 1: если активных меньше capacity → взять слот, counters['admitted'] += 1,
        #        вернуть 'admit'.
        # Шаг 2: иначе, если очередь короче queue_slots → отметить время прихода в
        #        _waiting, counters['queued'] += 1, вернуть 'queue'.
        # Шаг 3: иначе counters['rejected'] += 1 и вернуть 'reject'.
        raise NotImplementedError('TODO: решение о впуске по шагам')

    async def wait_turn(self, request_id: str) -> None:
        """Park one queued request until a service slot frees for it.

        Parameters
        ----------
        request_id : str
            Id placed into the waiting line by admit() ('queue' verdict).

        Returns
        -------
        None
            Returns on promotion by release(); таймаутом владеет вызывающий.
        """
        event = asyncio.Event()
        self._events[request_id] = event
        await event.wait()

    def release(self, request_id: str) -> None:
        """Free one service slot and immediately promote the oldest waiter, if any.

        Parameters
        ----------
        request_id : str
            Id of the finishing request, previously admitted to a service slot.

        Returns
        -------
        None
            Вызывается шлюзом в finally; пробуждение будит wait_turn().
        """
        with self._lock:
            self._active.discard(request_id)
            self._promote_one_locked()

    def abandon(self, request_id: str) -> None:
        """Remove one request from the waiting line after its deadline.

        Parameters
        ----------
        request_id : str
            Id previously placed into the waiting line by admit().

        Returns
        -------
        None
            Парная операция к wait_turn() при выходе за бюджет (504).
        """
        with self._lock:
            self._waiting.pop(request_id, None)
            if request_id in self._active:
                self._active.discard(request_id)
                self._promote_one_locked()
        self._events.pop(request_id, None)

    def _promote_one_locked(self) -> None:
        """Move the oldest waiting request into the freed service slot (lock held).

        Returns
        -------
        None
            FIFO по времени прихода; событие будит wait_turn().
        """
        for rid in self._waiting:
            del self._waiting[rid]
            self._active.add(rid)
            event = self._events.pop(rid, None)
            if event is not None:
                event.set()
            return


queue_probe = AdmissionQueue(capacity=2, queue_slots=1)
_verdicts = [queue_probe.admit(f'p{i}') for i in range(4)]
_rows = [dict(request_id=f'p{i}', verdict=v) for i, v in enumerate(_verdicts)]
import pandas as pd
display(pd.DataFrame(_rows).style.set_caption(
    f'ручная проверка решения: {queue_probe.counters} — два admit, один queue, '
    'один reject; слоты отпускаем ниже, чтобы не течь'))
for _rid in ('p0', 'p1'):
    queue_probe.release(_rid)
queue_probe.abandon('p2')

### Drill перегрузки: конверт против наблюдения

Вопрос клетки: совпадёт ли наблюдаемая доля 429 с конвертом «8 обслужены / 16
отказаны»? 24 клиента бьют одновременно; решение admit → queue → reject принимает
ВАШ `AdmissionQueue`, встроенный в шлюз вместе с валидацией. Оценка времени: ~3–5 с
(обслуживание выровнено объявленной арифметикой мока: 1.43 с на запрос из 64 токенов).

**Что вы должны увидеть:** таблицу наблюдение-против-конверта и подпись, сходится ли
конверт. При расхождении — не «подкрутить мок», а объяснить механизм (например,
медленный старт потоков на перегруженной машине).

In [ ]:
# Перегрузка: волна 24 клиентов в упор (S2). Оценка времени: ~3–5 с — обслуживание
# выровнено объявленной арифметикой мока (1.43 с на запрос из 64 токенов).
engine = MockEngine()
admission = AdmissionQueue(capacity=4, queue_slots=4)
stop_gateway(server, thread)
server, thread = start_gateway(
    make_gateway(engine, queue=admission, validate_fn=validate_request), PORT)

def _burst_client(cid: int) -> dict:
    """Send one completion as burst client cid and classify the HTTP answer.

    Parameters
    ----------
    cid : int
        Client index inside the burst (только для читаемости журнала).

    Returns
    -------
    dict
        Row: request_id, http_status, outcome (completed / rejected / other).
    """
    status, reply = http_request(BASE + '/v1/completions',
                                 body={'prompt': f'бурст {cid}', 'max_tokens': 64})
    outcome = {200: 'completed', 429: 'rejected'}.get(status, f'other:{status}')
    return dict(request_id=reply.get('request_id', f'c{cid}'),
                http_status=status, outcome=outcome)

with ThreadPoolExecutor(max_workers=24) as pool:
    _burst = list(tqdm(pool.map(_burst_client, range(24)), total=24,
                       desc='волна 24', unit='req'))
_counts = {k: sum(1 for r in _burst if r['outcome'] == k)
           for k in ('completed', 'rejected')}
_rows = [dict(outcome='completed (4 слота + 4 из очереди)',
              observed=_counts['completed'], envelope=8),
         dict(outcome='rejected 429', observed=_counts['rejected'], envelope=16)]
_ok = _counts['completed'] == 8 and _counts['rejected'] == 16
with open(os.path.join(RESULTS, 'overload-drill.json'), 'w', encoding='utf-8') as fh:
    json.dump(dict(rows=_burst, counts=_counts,
                   envelope=dict(admitted=4, queued=4, rejected=16)),
              fh, ensure_ascii=False, indent=1)
import pandas as pd
display(pd.DataFrame(_rows).style.set_caption(
    'бурст 24 при пуле 4+4: конверт '
    + ('СХОДИТСЯ' if _ok else 'РАСХОДИТСЯ — назовите механизм')
    + ' (overload-drill.json)'))

## Блок 3 — Честные ошибки: исход → код → действие

Карта исходов: invalid → 400; overloaded → 429 + `Retry-After`; engine_down → 503;
deadline → 504 с request id; всё остальное → 500 без внутренностей.
**Никогда не 200-with-error-body:** статус-код — первая правда, которую читает клиент;
200 с телом «на самом деле ошибка» молча ломает чужую retry-логику.

**Два drill'а по требованию** (клетка после функции): невалидный запрос и лежащий
движок. Доказательство — статус из настоящего HTTP и строка в журнале.

**Вопрос:** какой код получит запрос, который прождал в очереди дольше бюджета, —
и почему вместе с кодом едет request id?

In [ ]:
def map_engine_error(outcome: str, request_id: str) -> dict:
    """Map one terminal outcome to an honest HTTP answer for the client.

    Parameters
    ----------
    outcome : str
        Terminal outcome: 'invalid', 'overloaded', 'engine_down', 'deadline'
        или 'engine_bug'.
    request_id : str
        Gateway-assigned id; едет в теле каждой ошибки.

    Returns
    -------
    dict
        Keys: http_status (int), payload (dict с error и request_id),
        headers (dict, может быть пустым).
    """
    # Уровень B: скелет готов — соберите карту по шагам.
    # Шаг 1: 'invalid' → 400, payload с error и request_id.
    # Шаг 2: 'overloaded' → 429 и header Retry-After: '1'.
    # Шаг 3: 'engine_down' → 503.
    # Шаг 4: 'deadline' → 504; в payload — request_id, чтобы сбой в 3 ночи был объясним.
    # Шаг 5: всё остальное ('engine_bug') → 500 без деталей внутренностей.
    raise NotImplementedError('TODO: карта исход → код по шагам')


_rows = []
for _outcome in ('invalid', 'overloaded', 'engine_down', 'deadline', 'engine_bug'):
    _mapped = map_engine_error(_outcome, 'r-drill')
    _rows.append(dict(outcome=_outcome, http_status=_mapped['http_status'],
                      retry_after=_mapped.get('headers', {}).get('Retry-After', '—'),
                      error=_mapped['payload'].get('error', '')))
import pandas as pd
display(pd.DataFrame(_rows).style.set_caption(
    'карта исход → код → действие: ни один исход не притворяется успехом'))

In [ ]:
# Две поломки по требованию (S4): невалидный запрос и лежащий движок.
# Оценка времени: ~3 с (второй drill ждёт один объявленный шаг мока).
engine.down = True
_down_status, _down = http_request(BASE + '/v1/completions',
                                   body={'prompt': 'проверка 503', 'max_tokens': 8})
engine.down = False
_bad_status, _bad = http_request(BASE + '/v1/completions',
                                 body={'prompt': 'x' * 600, 'max_tokens': 8})
_rows = [dict(drill='невалидный запрос (промпт 600 символов)', observed=_bad_status,
              mapped=map_engine_error('invalid', 'r-drill')['http_status'],
              request_id=_bad.get('request_id', '—')),
         dict(drill='движок с флагом down', observed=_down_status,
              mapped=map_engine_error('engine_down', 'r-drill')['http_status'],
              request_id=_down.get('request_id', '—'))]
with open(os.path.join(RESULTS, 'error-drills.json'), 'w', encoding='utf-8') as fh:
    json.dump(_rows, fh, ensure_ascii=False, indent=1)
import pandas as pd
display(pd.DataFrame(_rows).style.set_caption(
    'два drill\'а: наблюдаемый статус совпал с картой; 200-with-error-body нет'))

## Структурированный вывод: схема на входе, валидация на выходе

Guided completion: в запросе едет `response_schema`; ответ обязан ей удовлетворять.
Нарушение — видимый путь вида `$.priority`, `$.score`, а не молчаливое приведение
типа. Мок возвращает объявленный синтетический JSON (настоящий grammar-constrained
decode живёт в движке); проверка схемы — ВАША функция, и она же защищает клиента от
тихой порчи данных.

**Фикстура нарушения (принцип «отладка — часть сюжета»):** заголовок
`x-mock-break-schema` заставляет мок вернуть `{'priority': 'срочно', 'score': '0.9'}`
— разберите, какие пути нарушены и почему это должно быть 422, а не 200.

**Вопрос:** что дороже — отклонить валидному клиенту его ответ или отдать 200 с
данными, которые не соответствуют обещанной схеме?

In [ ]:
GUIDED_SCHEMA = {'type': 'object',
                 'required': ['label', 'priority', 'score'],
                 'properties': {'label': {'type': 'string'},
                                'priority': {'enum': ['низкий', 'высокий']},
                                'score': {'type': 'number'}}}


def guided_completion(data: object, schema: dict) -> dict:
    """Validate one structured reply against the declared mini-schema.

    Parameters
    ----------
    data : object
        Parsed JSON reply from the model to check.
    schema : dict
        Mini-schema: type 'object', required names, per-field type/enum.

    Returns
    -------
    dict
        {'ok': True, 'violations': []} — или {'ok': False, 'violations': [...]}
        с путями вида '$.поле' для каждого нарушения.
    """
    # Уровень B: скелет готов.
    # Шаг 1: data должен быть dict — иначе единственное нарушение '$'.
    # Шаг 2: каждое имя из schema['required'] обязано присутствовать ('$.имя').
    # Шаг 3: для присутствующих полей сверите type ('string'/'number') и enum.
    # Шаг 4: верните dict(ok=..., violations=[...]) — путь и суть в каждой строке.
    raise NotImplementedError('TODO: мини-валидатор по шагам')


_good = {'label': 'оплата', 'priority': 'высокий', 'score': 0.9}
_bad = {'label': 'оплата', 'priority': 'срочно', 'score': '0.9'}
_rows = []
for _name, _data in (('валидный ответ', _good), ('нарушение схемы', _bad)):
    _check = guided_completion(_data, GUIDED_SCHEMA)
    _rows.append(dict(case=_name, ok=_check['ok'],
                      violations='; '.join(_check['violations']) or '—'))
import pandas as pd
display(pd.DataFrame(_rows).style.set_caption(
    'схема на входе, валидация на выходе: путь нарушения виден, тихих приведений нет'))

## Наблюдаемость: счётчики, которые вы планировали

Метрика, которую вы не планировали, станет метрикой, которую вы подделаете. Пять
чисел шлюза: accepted, rejected-429, cancelled, in-flight, отходы отмен. Каждое
кормит одно решение: масштабировать / отпустить нагрузку / искать брошенные
генерации / пересматривать SLO.

**Вопрос клетке:** снимок /metrics собирается по живому журналу шлюза — к этому
моменту `cancelled` будет честным нулём. Почему ноль — тоже данные, а не поломка
метрики?

**Зачем это в проде.** Когда шлюз ляжет в 3 ночи, разбираться будут по этим счётчикам: accepted отвечает «мы вообще обслуживаем?», rejected-429 — «дверь работает или сломана?», cancelled — «клиенты уходят до первого токена — где мы медлим?», in-flight — «сколько позиций занято прямо сейчас». Каждый счётчик должен иметь владельца и вопрос, на который он отвечает; счётчик без вопроса — мусор, который будет врать молча. Снимите снапшот до волн и после — разница между ними и есть ваша нагрузка, увиденная изнутри шлюза.


In [ ]:
def summarize_metrics(counters: dict, journal: Sequence[JournalRecord]) -> list[dict]:
    """Summarize gateway counters and journal into the /metrics snapshot rows.

    Parameters
    ----------
    counters : dict
        Admission counters: admitted, queued, rejected (из AdmissionQueue.counters).
    journal : sequence of JournalRecord
        Every gateway-observed request to date.

    Returns
    -------
    list of dict
        Rows: metric, value, feeds — какое решение питает это число.
    """
    # Уровень B: скелет готов.
    # Шаг 1: посчитайте исходы журнала: completed / invalid / overloaded /
    #        deadline / cancelled — по строке на исход.
    # Шаг 2: добавьте очередь: counters (admitted, queued, rejected) по строке.
    # Шаг 3: посчитайте wasted_tokens (сумма по журналу) и in_flight_now
    #        (строки без finished_ts).
    # Шаг 4: каждая строка — dict(metric=..., value=..., feeds=...), feeds одним
    #        предложением: какое решение примет человек, глядя на это число.
    raise NotImplementedError('TODO: снимок /metrics по шагам')


_rows = summarize_metrics(admission.counters, JOURNAL)
with open(os.path.join(RESULTS, 'metrics-snapshot.json'), 'w', encoding='utf-8') as fh:
    json.dump(_rows, fh, ensure_ascii=False, indent=1)
import pandas as pd
display(pd.DataFrame(_rows).style.set_caption(
    'снимок /metrics: каждое число кормит одно решение; ноль cancelled — тоже данные'))

## Блок 4 — Отмена: ушедший клиент перестаёт платить

Отключение клиента — это история preemption из Topic 06 в HTTP-одежде. Сначала
посмотрите на баг: клетка ниже «роняет» соединение посреди генерации БЕЗ наблюдателя
— движок досчитывает до конца, токены горят в никуда, журнал говорит «completed».

**Вопрос:** кто платит за эти токены и почему без учёта отходов эту цену не видит
никто — ни владелец, ни пейджер?

Потом вы напишете наблюдателя `watch_disconnect` — последняя и самая свободная
функция лабы (уровень C: только контракт в docstring). Она остановит генерацию,
пометит журнал «cancelled» и посчитает отходы. Сверите с прогнозом из блока 0.

In [ ]:
# Сознательно сломанная ситуация: наблюдатель не подключён, клиент «уходит».
# Оценка времени: ~2 с (ждём остаток объявленного прогона мока).
engine = MockEngine()
stop_gateway(server, thread)
server, thread = start_gateway(
    make_gateway(engine, queue=admission, validate_fn=validate_request), PORT)
_before = len(JOURNAL)
leave_after_send('/v1/completions',
                 json.dumps({'prompt': 'брошенный запрос',
                             'max_tokens': 64}).encode())
time.sleep(1.6)  # даём мок-движку досчитать объявленные 1.43 с без наблюдателя
_rec = JOURNAL[-1]
print(f"орфан: клиент ушёл сразу после отправки, движок досчитал: "
      f"outcome={_rec['outcome']}, сгенерировано {_rec['generated_tokens']} "
      f"токенов в никуда (полный прогон 1.43 с — объявлено)")

In [ ]:
async def watch_disconnect(request_id: str, engine_task: 'asyncio.Task',
                           disconnect: 'asyncio.Event', record: JournalRecord,
                           engine: MockEngine) -> dict:
    """Stop one in-engine generation when its client disconnects, and account waste.

    Контракт (уровень C — реализуйте тело целиком):
    1. дождитесь события disconnect;
    2. если engine_task уже завершён — отменять нечего: верните
       dict(request_id=..., cancelled=False, wasted_tokens=0), журнал не трогайте;
    3. иначе посчитайте wasted = engine.tokens_so_far(request_id), отмените
       engine_task и дождитесь его завершения (CancelledError — ожидаем);
    4. финализируйте журнал: record['outcome'] = 'cancelled',
       record['finished_ts'] = time.monotonic(), record['wasted_tokens'] = wasted;
    5. верните dict(request_id=..., cancelled=True, wasted_tokens=wasted).

    Parameters
    ----------
    request_id : str
        Id of the watched request (ключ учёта в движке).
    engine_task : asyncio.Task
        The in-engine generation task to cancel.
    disconnect : asyncio.Event
        Set by the gateway when the client is gone.
    record : JournalRecord
        Journal row of this request (мутируется на месте).
    engine : MockEngine
        The engine behind engine_task; источник tokens_so_far().

    Returns
    -------
    dict
        Keys: request_id, cancelled (bool), wasted_tokens (int).
    """
    raise NotImplementedError('TODO: уровень C — наблюдатель по контракту выше')


async def _probe_watcher() -> tuple[dict, JournalRecord]:
    """Run one watcher against a mock generation and return summary with record.

    Returns
    -------
    tuple of (dict, JournalRecord)
        Итог наблюдателя и финализированная строка журнала.
    """
    engine_probe = MockEngine()
    record = JournalRecord(request_id='probe-1', prompt_chars=10, max_tokens=64,
                           wave=-1, arrived_ts=time.monotonic(), finished_ts=None,
                           outcome='accepted', http_status=None,
                           generated_tokens=0, wasted_tokens=0)
    task = asyncio.create_task(engine_probe.generate('probe-1', 'x', 64, record))
    engine_probe.note_start('probe-1')
    disconnect = asyncio.Event()
    watcher = asyncio.create_task(
        watch_disconnect('probe-1', task, disconnect, record, engine_probe))
    await asyncio.sleep(0.10)  # клиент «уходит» через 0.10 с
    disconnect.set()
    summary = await watcher
    return summary, record

_summary, _record = await _probe_watcher()  # top-level await: ядро уже крутит цикл
print(f"наблюдатель: {_summary}; журнал: outcome={_record['outcome']}, "
      f"wasted={_record['wasted_tokens']} — генерация остановлена, отходы посчитаны")

In [ ]:
# Сверка контракта по HTTP: уходящий клиент против шлюза С наблюдателем.
# Оценка времени: ~2 с (заголовок задержки 0.30 с + остаток мока).
engine = MockEngine()
stop_gateway(server, thread)
server, thread = start_gateway(
    make_gateway(engine, queue=admission, validate_fn=validate_request,
                 watch_fn=watch_disconnect), PORT)
_before = len(JOURNAL)
leave_after_send('/v1/completions',
                 json.dumps({'prompt': 'отмена через HTTP',
                             'max_tokens': 64}).encode(),
                 headers={'x-disconnect-after-s': '0.30'})
time.sleep(0.8)  # даём наблюдателю и журналу финализировать строку
_recs = JOURNAL[_before:]
_rows = [dict(request_id=r['request_id'], outcome=r['outcome'],
              wasted_tokens=r['wasted_tokens'],
              generated_tokens=r['generated_tokens']) for r in _recs]
with open(os.path.join(RESULTS, 'cancel-drill.json'), 'w', encoding='utf-8') as fh:
    json.dump(_rows, fh, ensure_ascii=False, indent=1)
import pandas as pd
display(pd.DataFrame(_rows).style.set_caption(
    'drill отмены по HTTP: исход cancelled, отходы посчитаны — orphan-генерации нет '
    '(сверка с прогнозом блока 0)'))

In [ ]:
# Волны против шлюза: закрытый цикл из Topic 06 теперь бьёт в ваш HTTP-контракт.
# Каждая 4-я волна-клиент (cid % 4 == 3) «уходит» посреди генерации. Оценка времени:
# ~10–20 с на две волны; прогресс — tqdm (обслуживание мока: 0.79 с на запрос).
engine = MockEngine()
stop_gateway(server, thread)
server, thread = start_gateway(
    make_gateway(engine, queue=admission, validate_fn=validate_request,
                 watch_fn=watch_disconnect), PORT)
WAVES = (8, 24)

def _wave_client(cid: int, wave: int) -> int:
    """Send one completion as wave client cid and report the HTTP status.

    Parameters
    ----------
    cid : int
        Client index inside the wave; каждый четвёртый уходит посреди генерации
        через заголовок x-disconnect-after-s.
    wave : int
        Wave label; едет в журнал шлюза через заголовок x-wave.

    Returns
    -------
    int
        HTTP status; 0 — сокет умер вместе с уходящим клиентом (ожидаемо).
    """
    headers = {'x-wave': str(wave)}
    if cid % 4 == 3:
        headers['x-disconnect-after-s'] = '0.30'
    status, _reply = http_request(BASE + '/v1/completions',
                                  body={'prompt': f'волна {wave}, клиент {cid}',
                                        'max_tokens': 32},
                                  headers=headers)
    return status

for w in tqdm(WAVES, desc='волны', unit=' волна'):
    with ThreadPoolExecutor(max_workers=w) as pool:
        list(pool.map(lambda c: _wave_client(c, w), range(w)))

def _share(records: Sequence[JournalRecord], kind: str) -> str:
    """Format one outcome share over the wave attempts.

    Parameters
    ----------
    records : sequence of JournalRecord
        Journal rows of one wave (знаменатель — каждая попытка).
    kind : str
        Outcome name: 'completed', 'overloaded' или 'cancelled'.

    Returns
    -------
    str
        Доля исхода в процентах попыток волны.
    """
    if not records:
        return '—'
    return f"{sum(1 for r in records if r['outcome'] == kind) / len(records):.0%}"

_rows = []
for w in WAVES:
    rs = [r for r in JOURNAL if r['wave'] == w]
    _rows.append(dict(wave=w, attempts=len(rs), completed=_share(rs, 'completed'),
                      rejected_429=_share(rs, 'overloaded'),
                      cancelled=_share(rs, 'cancelled')))
import pandas as pd
display(pd.DataFrame(_rows).style.set_caption(
    'волны через шлюз (журнал, знаменатель — каждая попытка): 429 там, где предсказывал '
    'блок 0; cancelled только с наблюдателем; числа mock — объявленная синтетика'))

In [ ]:
# Вопрос: как ваши решения на двери распределились по волнам? Доли исходов
# (completed / rejected / cancelled) — ответ, прочитанный из журнала (mock, объявлено).
import matplotlib
try:
    get_ipython()  # интерактивный фронтенд: инлайн-бэкенд, график виден
except NameError:
    matplotlib.use('Agg')  # headless-запуск: только файлы
import matplotlib.pyplot as plt
_COLORS = dict(completed='#146957', overloaded='#af372b', cancelled='#89540a')
_waves = sorted({r['wave'] for r in JOURNAL if r['wave'] > 0})
fig, ax = plt.subplots(figsize=(7, 4.2))
bottoms = [0.0] * len(_waves)
for kind in ('completed', 'overloaded', 'cancelled'):
    shares = []
    for w in _waves:
        rs = [r for r in JOURNAL if r['wave'] == w]
        shares.append(sum(1 for r in rs if r['outcome'] == kind) / max(1, len(rs)))
    ax.bar([str(w) for w in _waves], shares, bottom=bottoms, label=kind,
           color=_COLORS[kind])
    for i, (b, s) in enumerate(zip(bottoms, shares)):
        if s >= 0.06:
            ax.annotate(f'{s * 100:.0f}%', (i, b + s / 2), ha='center', va='center',
                        color='white', fontsize=9)
    bottoms = [b + s for b, s in zip(bottoms, shares)]
ax.axhline(1.0, color='#4b6070', linestyle=':', lw=1.2)
ax.annotate('100% = каждая попытка волны', xy=(0.99, 1.0),
            xycoords=('axes fraction', 'data'), ha='right', va='bottom',
            fontsize=8, color='#4b6070')
ax.set_xlabel('волна (закрытый цикл, клиентов)')
ax.set_ylabel('доля попыток')
ax.set_title('Волны против шлюза: решения из журнала (mock, объявленная синтетика)')
ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, 'wave-shares.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Заморозка контракта шлюза: SLO + мемо. Значения SLO — ОБЪЯВЛЕННЫЙ выбор студента
# (не измерение); колено перегрузки — из экономики очереди блока 2 (≈17 req/s).
GATEWAY_SLO = dict(p95_in_gateway_s=2.0, error_budget_share=0.01,
                   rejection_policy='429 + Retry-After при пуле 4+4; повтор с паузой')
MEMO = dict(
    audience='продукт + дежурный (читают за 30 секунд)',
    promise='TODO: одно предложение — что шлюз обещает при λ до 17 req/s '
            '(колено блока 2) и что делает дальше',
    refusal='TODO: одно предложение — как шлюз отказывает, когда слотов нет, '
            'и что клиент видит вместе с отказом (request id, Retry-After)',
    bridge='T09: шлюз уезжает в неизменяемый образ — контракт переживёт смену машины')
_open = [k for k, v in MEMO.items() if str(v).startswith('TODO')]
with open(os.path.join(RESULTS, 'gateway-memo.json'), 'w', encoding='utf-8') as fh:
    json.dump(dict(slo=GATEWAY_SLO, memo=MEMO), fh, ensure_ascii=False, indent=1)
print(f'контракт заморожен (gateway-memo.json); мемо: '
      f'{len(MEMO) - len(_open)}/{len(MEMO)} абзацев готово — допишите TODO до сдачи')

## Что вы построили — и чего эта лаба не доказывает

**Артефакт:** один растущий шлюз — от голого v0-пропуска до контракта с капами,
ограниченной очередью, картой ошибок, отменой, структурированным выводом и волнами —
плюс журнал в `results/mock/attempt-001/`: api-contract.json, overload-drill.json,
error-drills.json, metrics-snapshot.json, cancel-drill.json, wave-shares.png,
gateway-memo.json, manifest.json с sha256.

**Чему научились:** переводить каждый механизм Topic 06 в решение API; предсказывать
поведение шлюза до кода и сверять прогнозы с журналом; отвечать перегрузке 429 вместо
вечной очереди; останавливать брошенные генерации и считать отходы; никогда не
отвечать 200-with-error-body; проверять структурированный вывод против схемы; видеть
собственные решения admission в журнале волн.

**Чего НЕ доказывает эта лаба:** ручки мока — объявленная синтетика, не vLLM;
задержка обнаружения отключения на моке — лучший случай, реальные сети шумнее; два
drill'а покрывают названные режимы, а не весь каталог отказов; шлюз один —
горизонтальное масштабирование за пределами темы; живой vLLM за шлюзом —
демонстрация владельца с закрытыми доказательствами.

**Куда дальше:** Topic 09 пакует этот шлюз в неизменяемый образ: контракт, который
переживает смену машины. Пропуск, который вы научились выдавать и честно отказывать,
станет артефактом, который можно перевезти.

In [ ]:
artifacts = {}
for name in ('api-contract.json', 'overload-drill.json', 'error-drills.json',
             'metrics-snapshot.json', 'cancel-drill.json', 'wave-shares.png',
             'gateway-memo.json'):
    path = os.path.join(RESULTS, name)
    if os.path.exists(path):
        with open(path, 'rb') as fh:
            artifacts[name] = hashlib.sha256(fh.read()).hexdigest()
with open(os.path.join(RESULTS, 'manifest.json'), 'w', encoding='utf-8') as fh:
    json.dump(dict(profile=PROFILE, attempt=ATTEMPT, artifacts=artifacts),
              fh, ensure_ascii=False, indent=1)
print(f'{len(artifacts)} артефактов хэшировано в manifest.json — журнал пропуска закрыт')

In [ ]:
# Штатная остановка СВОЕГО процесса: should_exit + join (правило курса — только owned).
stop_gateway(server, thread)
_alive = False
try:
    urllib.request.urlopen(BASE + '/health', timeout=2)
    _alive = True
except (urllib.error.URLError, OSError):
    pass
print(f'teardown: шлюз на порту {PORT} остановлен штатно; /health молчит: {not _alive}')